# TODO: You must set checkpoint_directory to where you want to save and load the model

In [ ]:
checkpoint_directory ="/Users/jameswalker/Programming/Checkpoints"

In [ ]:
import ray 

ray.shutdown()
ray.init() 

2025-04-14 13:42:58,217	INFO worker.py:1852 -- Started a local Ray instance.


Python version:,3.9.17
Ray version:,2.44.1


In [3]:
from ray.rllib.algorithms.ppo import PPOConfig
from ray.rllib.core.rl_module.default_model_config import DefaultModelConfig
from tqdm import tqdm 
from metadrive.envs.metadrive_env import MetaDriveEnv
from metadrive.envs.safe_metadrive_env import SafeMetaDriveEnv

# Not necessary to use this callback, since use_render defaults to False anyways
# def env_creator(env_config):
#     env = MetaDriveEnv(config={"use_render": False})
#     return env 

# This register_env did not work with the callback. I had to hardcode the environment to MetaDriveEnv in the config
# register_env('metadrive-env', env_creator)

# Not sure if sgd_minibatch_size is being handled properly, but train_batch_size and worker_num are for sure
# worker_num = 1
# train_batch_size = int(1024)
# sgd_minibatch_size = max(256, int(train_batch_size/10))

# https://docs.ray.io/en/latest/rllib/package_ref/doc/ray.rllib.algorithms.algorithm_config.AlgorithmConfig.html
config = ( PPOConfig()
    .environment(
        # MetaDriveEnv,
        SafeMetaDriveEnv,
        disable_env_checking=True # TODO Is this right?
        )
    .env_runners(
        num_env_runners=1, 
        num_cpus_per_env_runner=1,
        num_gpus_per_env_runner=0
        )
    .framework('torch')
    # The below .learners comes from the migration guide when they talk about no GPU but multi-CPU
    .learners(
        num_learners=2,
        num_cpus_per_learner=1,
        num_gpus_per_learner=0,
    )
    .training(
        # train_batch_size=train_batch_size, # Deprecated
        train_batch_size_per_learner=int(512), # Replacement for train_batch_size
        # sgd_minibatch_size=sgd_minibatch_size, # This got deprecated
        # minibatch_size=sgd_minibatch_size, # This might be the replacement
        lr=1e-4,
        entropy_coeff=0.001,
    )    
    # .rl_module will be necessary for defining custom multi-agent models
    .rl_module(
        # Use a non-default 32,32-stack with ReLU activations.
        # This is for PPO since it uses multi-layer perceptrons (MLP)
        model_config=DefaultModelConfig(
            fcnet_hiddens=[32, 32],
            fcnet_activation="relu",
        )
    )
    # The below, commented .training illustrates grid search for learning rate,
    # comes from https://github.com/ray-project/ray/blob/master/rllib/examples/gpus/fractional_gpus_per_learner.py
    # .training(
    #     lr=tune.grid_search([0.005, 0.003, 0.001, 0.0001])
    # )
    .evaluation(
        evaluation_num_env_runners=1, 
        evaluation_duration=10000,
        # TODO evaluation_interval?
        # WARNING algorithm_config.py:4593 -- You have specified 1 evaluation workers, but your `evaluation_interval` is 0 or None! Therefore, evaluation doesn't occur automatically with each call to `Algorithm.train()`. Instead, you have to call `Algorithm.evaluate()` manually in order to trigger an evaluation run.
    )
    # .resources is only for setting num_cpus_for_main_process, I don't think we need to mess with it
    # .resources(
    #     num_gpus=int(0),  # This is deprecated
    #     num_cpus_for_main_process=int(1), # Defaults to 1
    # )  
    .debugging(log_level='ERROR') # INFO, DEBUG, ERROR, WARN
)



In [ ]:
# config.build() is deprecated, use build_algo() instead
algo = config.build_algo()
# print(f"Type of algo: {type(algo)}")


2025-04-14 13:43:06,910	WARNING algorithm_config.py:4593 -- You have specified 1 evaluation workers, but your `evaluation_interval` is 0 or None! Therefore, evaluation doesn't occur automatically with each call to `Algorithm.train()`. Instead, you have to call `Algorithm.evaluate()` manually in order to trigger an evaluation run.
2025-04-14 13:43:06,914	WARNING algorithm_config.py:4704 -- You are running PPO on the new API stack! This is the new default behavior for this algorithm. If you don't want to use the new API stack, set `config.api_stack(enable_rl_module_and_learner=False,enable_env_runner_and_connector_v2=False)`. For a detailed migration guide, see here: https://docs.ray.io/en/master/rllib/new-api-stack-migration-guide.html
/Users/jameswalker/miniconda3/lib/python3.9/site-packages/ray/rllib/algorithms/algorithm.py:512: RayDeprecationWarning: This API is deprecated and may be removed in future Ray releases. You could suppress this warning by setting env variable PYTHONWARNING

Type of algo: <class 'ray.rllib.algorithms.ppo.ppo.PPO'>


# TRAIN THE ALGORITHM
100 training iterations took about 2 minutes on my Mac with M1 chip
If setting it up on a server, just try 100 iterations and see if you can save and load the Algorithm instance. After that try 10,000 iterations

In [ ]:
# iteration_num = 10_000
iteration_num = 100
# iteration_num = 5
# iteration_num = 1
for iteration in tqdm(range(iteration_num)):
    result = algo.train()
    # For seeing all the metrics we can use in the result dict():
    # for key in result.keys():
    #     print(f"Key {key}: {result[key]}")
    # TODO Are these good metrics below? Not sure exactly what either means
    if iteration % (iteration_num/10) == 0:
        print(f"Iteration {iteration} Episode Length Mean: {result['env_runners']['episode_len_mean']}")
        print(f"Iteration {iteration} Agent Episode Returns Mean: {result['env_runners']['agent_episode_returns_mean']}")


  0%|          | 0/100 [00:00<?, ?it/s](SingleAgentEnvRunner pid=52570) [INFO] Assets version: 0.4.3
(SingleAgentEnvRunner pid=52570) [INFO] Known Pipes: CocoaGraphicsPipe
(SingleAgentEnvRunner pid=52570) [INFO] Start Scenario Index: 0, Num Scenarios : 100
  1%|          | 1/100 [00:02<03:32,  2.15s/it]

Iteration 0 Episode Length Mean: 1000.0
Iteration 0 Agent Episode Returns Mean: {'default_agent': 13.321248819873553}


 11%|█         | 11/100 [00:21<03:11,  2.15s/it]

Iteration 10 Episode Length Mean: 491.99
Iteration 10 Agent Episode Returns Mean: {'default_agent': 6.827588978064642}


 21%|██        | 21/100 [00:44<03:11,  2.42s/it]

...

 81%|████████  | 81/100 [03:04<00:42,  2.23s/it]

Iteration 80 Episode Length Mean: 72.48
Iteration 80 Agent Episode Returns Mean: {'default_agent': 40.14615582379084}


 91%|█████████ | 91/100 [03:26<00:21,  2.34s/it]

Iteration 90 Episode Length Mean: 71.31
Iteration 90 Agent Episode Returns Mean: {'default_agent': 41.75060255335772}


100%|██████████| 100/100 [03:46<00:00,  2.27s/it]


# Save Algorithm instance
"An Algorithm instance typically holds a neural network for computing actions, called the policy, the RL environment you want to optimize against, a loss function, an optimizer, and some code describing the algorithm’s execution logic, like determining when to collect samples, when to update your model, etc."

We can checkpoint an Algorithm to save its state then load it into a new Algorithm instance at a later date.

In [6]:
# Save Algorithm checkpoint.
algo.save_to_path(checkpoint_directory)

# Display the Algorithm RLModule to check we loaded the same RLModule later on
print(type(algo))
algo.get_module()

<class 'ray.rllib.algorithms.ppo.ppo.PPO'>


DefaultPPOTorchRLModule(
  (encoder): TorchActorCriticEncoder(
    (encoder): TorchMLPEncoder(
      (net): TorchMLP(
        (mlp): Sequential(
          (0): Linear(in_features=259, out_features=32, bias=True)
          (1): ReLU()
          (2): Linear(in_features=32, out_features=32, bias=True)
          (3): ReLU()
        )
      )
    )
  )
  (pi): TorchMLPHead(
    (net): TorchMLP(
      (mlp): Sequential(
        (0): Linear(in_features=32, out_features=4, bias=True)
      )
    )
  )
)

# Loading the Algorithm instance

In [7]:
from ray.rllib.core.rl_module.rl_module import RLModule
from ray.rllib.algorithms.algorithm import Algorithm
import os
from ray.rllib.core import DEFAULT_MODULE_ID

algo_loaded = Algorithm.from_checkpoint(checkpoint_directory)

# Compare the Algorithm we loaded with the one we stored earlier
print(type(algo_loaded))
algo_loaded.get_module()

/Users/jameswalker/miniconda3/lib/python3.9/site-packages/ray/rllib/algorithms/algorithm.py:512: RayDeprecationWarning: This API is deprecated and may be removed in future Ray releases. You could suppress this warning by setting env variable PYTHONWARNINGS="ignore::DeprecationWarning"
`UnifiedLogger` will be removed in Ray 2.7.
  return UnifiedLogger(config, logdir, loggers=None)
/Users/jameswalker/miniconda3/lib/python3.9/site-packages/ray/tune/logger/unified.py:53: RayDeprecationWarning: This API is deprecated and may be removed in future Ray releases. You could suppress this warning by setting env variable PYTHONWARNINGS="ignore::DeprecationWarning"
The `JsonLogger interface is deprecated in favor of the `ray.tune.json.JsonLoggerCallback` interface and will be removed in Ray 2.7.
  self._loggers.append(cls(self.config, self.logdir, self.trial))
/Users/jameswalker/miniconda3/lib/python3.9/site-packages/ray/tune/logger/unified.py:53: RayDeprecationWarning: This API is deprecated and m

<class 'ray.rllib.algorithms.ppo.ppo.PPO'>


DefaultPPOTorchRLModule(
  (encoder): TorchActorCriticEncoder(
    (encoder): TorchMLPEncoder(
      (net): TorchMLP(
        (mlp): Sequential(
          (0): Linear(in_features=259, out_features=32, bias=True)
          (1): ReLU()
          (2): Linear(in_features=32, out_features=32, bias=True)
          (3): ReLU()
        )
      )
    )
  )
  (pi): TorchMLPHead(
    (net): TorchMLP(
      (mlp): Sequential(
        (0): Linear(in_features=32, out_features=4, bias=True)
      )
    )
  )
)

(_WrappedExecutable pid=52668) Setting up process group for: env:// [rank=0, world_size=2]
(_WrappedExecutable pid=52668) 2025-04-14 13:48:17,258	WARNING deprecation.py:50 -- DeprecationWarning: `RLModule(config=[RLModuleConfig object])` has been deprecated. Use `RLModule(observation_space=.., action_space=.., inference_only=.., model_config=.., catalog_class=..)` instead. This will raise an error in the future!


https://github.com/ray-project/ray/tree/master/rllib/examples/inference

In [8]:
print(type(algo_loaded.get_module()))

<class 'ray.rllib.algorithms.ppo.torch.default_ppo_torch_rl_module.DefaultPPOTorchRLModule'>


# How to debug AssertionError
AssertionError: Can not call this API after engine initialization!

Run the cell below. It should fix this error. This error gets raised if you press 'ESC' and shutdown the Environment but the renderer doesn't properly shutdown.

In [58]:
env.close()

# Testing your trained model
This runs for 300 steps. My model was super bad so it never got all the checkpoints, so I'm not sure what happens if all checkpoints are collected.

In [59]:
from ray.rllib.algorithms.ppo.ppo_catalog import PPOCatalog
import torch

env = SafeMetaDriveEnv({
    "use_render": True,
    "manual_control": False,
    "vehicle_config": {
        "show_dest_mark": True,
        "show_navi_mark": True,
        # Whether to draw a line from current vehicle position to the designation point
        "show_line_to_dest":True,
        # Whether to draw a line from current vehicle position to the next navigation point
        "show_line_to_navi_mark":True,
        # Whether to draw left / right arrow in the interface to denote the navigation direction
        "show_navigation_arrow":True,
        # If set to True, the vehicle will be in color green in top-down renderer or MARL setting
        "use_special_color":True,
    },
    })
module = algo_loaded.get_module()
action_dist_class = module.get_inference_action_dist_cls()
obs, info = env.reset()
total_cost = 0
try:
    for i in range(1, 300):
        if i % 100 == 0:
            print(f"Iteration {i}")
        fwd_ins = {"obs": torch.Tensor([obs])}
        fwd_outputs = module.forward_exploration(fwd_ins)
        # This can be either deterministic or stochastic distribution.
        action_dist = action_dist_class.from_logits(
            fwd_outputs["action_dist_inputs"]
        )
        action = action_dist.sample()[0].numpy()
        obs, reward, terminated, truncated, info = env.step(action)
        total_cost += info["cost"]
        ret = env.render(
            # mode="topdown", 
            mode="birdview", 
            screen_record=True,
            window=False,
            screen_size=(600, 600), 
            # camera_position=(50, 50),
            text={
                "cost": total_cost,
                "seed": env.current_seed,
                "reward": reward,
                "total_cost": info["total_cost"],
            }
        )
        if info["crash_vehicle"]:
            print("crash_vehicle:cost {}, reward {}".format(info["cost"], reward))
        if info["crash_object"]:
            print("crash_object:cost {}, reward {}".format(info["cost"], reward))
        # if terminated or truncated:
        #     total_cost = 0
        #     print("done_cost:{}".format(info["cost"]), "done_reward;{}".format(r))
        #     print("Reset")
        #     env.reset()
    print(f"Top down renderer: {env.top_down_renderer}")
    env.top_down_renderer.generate_gif()
finally:
    env.close()
print("Finished")

[INFO] Environment: SafeMetaDriveEnv
[INFO] MetaDrive version: 0.4.3
[INFO] Sensors: [lidar: Lidar(), side_detector: SideDetector(), lane_line_detector: LaneLineDetector(), main_camera: MainCamera(1200, 900), dashboard: DashBoard()]
[INFO] Render Mode: onscreen
[INFO] Horizon (Max steps per agent): 1000
[INFO] Assets version: 0.4.3
[INFO] Known Pipes: CocoaGraphicsPipe
[INFO] Start Scenario Index: 0, Num Scenarios : 100


Iteration 100
Iteration 200
Top down renderer: <metadrive.engine.top_down_renderer.TopDownRenderer object at 0x5706caf70>
Finished
